# 01 - M5 Exploratory Data Analysis

Working notebook for the exploration pass. It walks the six question groups in
[`docs/data-exploration-questions.md`](../docs/data-exploration-questions.md); record findings back in
[`docs/data-dictionary.md`](../docs/data-dictionary.md).

**Not the deliverable** — this is `notebooks/` scratch. The pipeline lives in `src/`.

**Memory strategy:** the sales files are wide (~30k series x ~1.9k day-columns, ~120 MB each). We keep
them wide and compute per-series / per-day summaries *without a full melt* (a full melt to long is ~58M
rows — that's the Spark justification for Feature A, not something to do in pandas here). We melt only a
single slice when we need actual time-series curves.

## 0. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)

RAW = Path("..") / "data" / "raw"
assert RAW.exists(), f"expected {RAW.resolve()} to exist"
sorted(p.name for p in RAW.glob("*.csv"))

In [ ]:
calendar = pd.read_csv(RAW / "calendar.csv", parse_dates=["date"])
prices   = pd.read_csv(RAW / "sell_prices.csv")
# Use the evaluation file: it has the extra 28 days (d_1..d_1941).
sales    = pd.read_csv(RAW / "sales_train_evaluation.csv")

id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
d_cols  = [c for c in sales.columns if c.startswith("d_")]
print(f"sales: {sales.shape} | day-columns: {len(d_cols)} ({d_cols[0]}..{d_cols[-1]})")
print(f"calendar: {calendar.shape} | prices: {prices.shape}")

In [ ]:
calendar.head()

In [ ]:
prices.head()

In [ ]:
sales.head()

## 1. Shape & hierarchy
*How many series; how does cats -> depts -> items -> stores -> states fan out; date span; balanced panel?*

In [ ]:
print("series (rows):", len(sales))
for c in ["cat_id", "dept_id", "item_id", "store_id", "state_id"]:
    print(f"  {c:9s}: {sales[c].nunique():>5d} unique")

print("\nstores per state:")
print(sales.groupby("state_id")["store_id"].nunique())
print("\ndepts per category:")
print(sales.groupby("cat_id")["dept_id"].nunique())

In [ ]:
# Date span: map the d_ index to real dates via calendar.
cal_days = calendar.set_index("d")["date"]
print("first day:", d_cols[0],  "->", cal_days.get(d_cols[0]))
print("last  day:", d_cols[-1], "->", cal_days.get(d_cols[-1]))
print("calendar spans:", calendar['date'].min(), "to", calendar['date'].max(),
      f"({len(calendar)} rows; {len(calendar) - len(d_cols)} future days for the horizon)")
# Balanced panel? The wide matrix has no gaps by construction (every series has every d_ column).
# The real question is intermittency (zeros) vs true absence — see section 2 and the price note in 3.
print("any nulls in the day matrix?", bool(sales[d_cols].isna().any().any()))

## 2. The target — intermittency & seasonality
*Zero-inflation overall and by category; is it uniform or concentrated; weekly + annual seasonality?*

**Zero-Share**: fraction of days an item doesn't sell.

In [ ]:
# Per-series zero share (fraction of days with 0 sales), then summarise by category.
zero_share = (sales[d_cols] == 0).mean(axis=1)
sales_meta = sales[id_cols].copy()
sales_meta["zero_share"] = zero_share

print("overall zero-share: mean %.3f | median %.3f" % (zero_share.mean(), zero_share.median()))
print("\nzero-share by category (mean):")
print(sales_meta.groupby("cat_id")["zero_share"].agg(["mean", "median", "max"]).round(3))
# TODO: does one category look far more modellable (denser) than the others?

In [ ]:
# Aggregate daily sales curve WITHOUT a full melt: sum each d_ column across all series.
daily_total = sales[d_cols].sum(axis=0)              # Series indexed by d_1..d_N
daily_total.index = daily_total.index.map(cal_days)   # -> real dates
daily_total = daily_total.sort_index()

ax = daily_total.plot(figsize=(13, 4), title="Total units sold per day (all series)")
ax.set_xlabel("date"); ax.set_ylabel("units")
# TODO: eyeball annual seasonality + the Christmas closures (Walmart shut -> hard zeros).

In [ ]:
# Weekly seasonality: average total by weekday.
dow = daily_total.groupby(daily_total.index.dayofweek).mean()
dow.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
print(dow.round(0))
# TODO: confirm the weekend lift — it drives the seasonal-naive baseline (period = 7).

## 3. Price — does the DiD causal layer have anything to bite on?
*The decision-driver. Do prices move within a store-item; are drops sharp & datable; is there a control group?*

In [ ]:
# What fraction of store-item series ever change price?
n_prices = prices.groupby(["store_id", "item_id"])["sell_price"].nunique()
print("store-item series with price data:", len(n_prices))
print("ever change price: %.1f%%" % (100 * (n_prices > 1).mean()))
print(n_prices.value_counts().sort_index().head(10))
# TODO: enough movement to build a DiD around? If most are flat, the treated pool is small.

In [ ]:
# Find sharp price DROPS (datable steps) — candidate DiD interventions.
wk_order = calendar[["wm_yr_wk", "date"]].drop_duplicates().sort_values("date")
px = prices.merge(wk_order.groupby("wm_yr_wk")["date"].min().rename("week_start"),
                  on="wm_yr_wk", how="left")
px = px.sort_values(["store_id", "item_id", "week_start"])
px["pct_change"] = px.groupby(["store_id", "item_id"])["sell_price"].pct_change()

drops = px[px["pct_change"] <= -0.15]   # >=15% cut in one week; tune the threshold
print(f"{len(drops)} week-on-week cuts >=15%%, across {drops.groupby(['store_id','item_id']).ngroups} store-items")
drops.sort_values("pct_change").head(15)[["store_id", "item_id", "week_start", "sell_price", "pct_change"]]
# TODO: pick one clean, datable cut. Then find same-store items in the same dept that did NOT cut = control.

### 3b — From "cuts exist" to "here is a clean intervention"

The `head()` above is misleading *by construction*: sorting by the most-negative `pct_change` surfaces the
**artifact tail** — prices collapsing to ~$0.01 (−99%). Those are clearances / data glitches, not promotions,
and a DiD built on them is meaningless. The biggest number is almost never the cleanest example.

A usable DiD intervention needs a **moderate, datable, stable step** (e.g. $5.00 → $3.50, then flat) so there
is a clean *before/after* boundary to compare. The next cell finds those by:

1. **Plotting the cut distribution** so the penny-price spike near −1.0 is visible (see the trap, don't just trust the filter).
2. **Reworking the price path into constant-price "runs"** — a run is a stretch of weeks at one price. A cut becomes
   "one run ended, a lower run began," which lets us measure how *stable* the price was on each side of the step.
3. **Keeping only clean FOODS candidates:** cut of 15–45% (believable promo, not an artifact), landing on a non-penny
   price, stable for ≥4 weeks *before and after* (a genuine step, not a one-week blip). FOODS matches the §2 scope lean.

Output is a shortlist to pick **one** treated item from — then build its control (same store, same dept, no cut
in-window) and check the pre-trends are parallel.


In [ ]:
# WHY THIS CELL: the head() above sorts by the *most negative* pct_change, which surfaces the ARTIFACT
# TAIL — prices collapsing to ~$0.01 (-99%). Those are clearances / data-entry glitches, not promotions,
# and are useless for a DiD. A valid DiD needs a *moderate, datable, stable* cut (e.g. $5.00 -> $3.50,
# then flat) with a clean before/after boundary. So we (1) plot the cut distribution to SEE the artifact
# spike near -1.0, and (2) rework the price path into constant-price RUNS and keep only clean FOODS steps.
import matplotlib.pyplot as plt

# 1) Collapse each store-item price path into constant-price RUNS (a run = consecutive weeks at one price).
px = px.sort_values(["store_id", "item_id", "week_start"]).copy()
grp_key = ["store_id", "item_id"]
px["price_changed"] = px["sell_price"].ne(px.groupby(grp_key)["sell_price"].shift())  # True at each new price
px["run_id"] = px.groupby(grp_key)["price_changed"].cumsum()

runs = (px.groupby(grp_key + ["run_id"])
          .agg(price=("sell_price", "first"),
               run_len=("sell_price", "size"),      # how many weeks this price held
               start=("week_start", "first"))       # the datable week the step happened
          .reset_index())
# Each run's predecessor = the price/length just before this step (NaN for a series' first run).
runs["prev_price"] = runs.groupby(grp_key)["price"].shift()
runs["prev_len"]   = runs.groupby(grp_key)["run_len"].shift()
runs["drop_pct"]   = runs["price"] / runs["prev_price"] - 1
runs["cat"]        = runs["item_id"].str.split("_").str[0]

# 2) SEE the artifact tail: distribution of all price cuts.
cuts = runs.loc[runs["drop_pct"] < 0, "drop_pct"]
ax = cuts.plot(kind="hist", bins=60, figsize=(10, 3),
               title="Price-cut magnitudes (spike near -1.0 = penny-price artifacts, NOT promotions)")
ax.set_xlabel("drop_pct"); plt.show()
print("cuts >=50%% deep (artifact-suspect): %5d | cuts 15-45%% (promo-plausible): %5d"
      % (int((cuts <= -0.50).sum()), int(cuts.between(-0.45, -0.15).sum())))

# 3) FILTER to clean FOODS DiD candidates: moderate cut, non-penny price, stable >=K weeks either side.
K = 4  # weeks of price stability required before AND after the step (a real step, not a one-week blip)
cand = runs[(runs["cat"] == "FOODS") &
            (runs["drop_pct"].between(-0.45, -0.15)) &          # believable promo depth
            (runs["prev_len"] >= K) & (runs["run_len"] >= K) &  # stable step, not churn
            (runs["price"] >= 1.0)].copy()                      # non-penny -> not a clearance/glitch
cand["from_to"] = cand["prev_price"].round(2).astype(str) + " -> " + cand["price"].round(2).astype(str)
print(f"\n{len(cand)} clean FOODS candidates (>= {K}w stable each side, 15-45%% cut, price >= $1)")
cand.sort_values("drop_pct")[["store_id", "item_id", "start", "from_to",
                              "drop_pct", "prev_len", "run_len"]].head(20)
# TODO: pick ONE. Then build its control = same store, same dept, no cut in-window; check parallel pre-trends.

### 3c — Verify one candidate: the parallel-trends check

We pick **`FOODS_3_822` @ `TX_2`, cut 2015-08-29 ($4.28 → $2.50)** and pressure-test it before committing. Two
questions in one plot:

1. **Is it dense enough to model?** (the §2 lesson — a mostly-zero item has no learnable signal). We print its
   pre-cut mean units/week and zero-week share.
2. **Do treated and control share a parallel pre-trend?** This is *the* DiD identifying assumption. Controls are
   **same store, same department, price-stable around the cut** (no ≥15% cut within ±8 weeks). We index both series
   to their pre-cut mean (=100) so we compare *trajectories*, not levels (the treated item and the control average
   sell different absolute volumes; DiD only cares that they *move* together).

**How to read it:** look *left* of the dashed line. If the two lines track each other before the cut, parallel-trends
is credible and any post-cut **gap** (treated pulling above control) is the causal price effect. If they already
diverge before the cut, this control set is invalid — swap controls or pick another treated item.


In [ ]:
# --- Verify the DiD candidate: FOODS_3_822 @ TX_2, cut 2015-08-29 ($4.28 -> $2.50, -42%). ---
# Two checks in ONE plot: (a) does the treated item sell densely enough to model (the section-2 lesson),
# and (b) do treated + its same-store/same-dept controls share a PARALLEL PRE-TREND (the DiD assumption).
STORE, DEPT, TREATED = "TX_2", "FOODS_3", "FOODS_3_822"
CUT = pd.Timestamp("2015-08-29")
WIN = pd.Timedelta("56D")   # +/-8 weeks: a control must be price-stable in this window around the cut

# 1) Melt ONLY the TX_2 / FOODS_3 slice to long (small), then aggregate to weekly units per item.
sl = sales[(sales["store_id"] == STORE) & (sales["dept_id"] == DEPT)]
long = sl.melt(id_vars=["item_id"], value_vars=d_cols, var_name="d", value_name="units")
long["date"] = long["d"].map(cal_days)
long["week"] = long["date"].dt.to_period("W").dt.start_time
weekly = long.groupby(["item_id", "week"])["units"].sum().reset_index()

# 2) Controls = same store, same dept, NOT the treated item, and NO >=15% price cut near the event.
contaminated = set(runs[(runs["store_id"] == STORE) &
                        (runs["item_id"].str.startswith(DEPT + "_")) &
                        (runs["drop_pct"] <= -0.15) &
                        (runs["start"].between(CUT - WIN, CUT + WIN))]["item_id"])
controls = sorted(set(sl["item_id"]) - {TREATED} - contaminated)
print(f"controls: {len(controls)} same-store/same-dept items, price-stable near the cut")

# 3) Density check on the treated item (is it worth modelling at all?).
t = weekly[weekly["item_id"] == TREATED].set_index("week")["units"].sort_index()
pre = t[t.index < CUT]
print(f"treated pre-cut: mean {pre.mean():.1f} units/week | zero-weeks {100 * (pre == 0).mean():.0f}%")

# 4) Index treated and control-average to their PRE-CUT mean (=100): compares *trajectories*, not levels.
c = weekly[weekly["item_id"].isin(controls)].groupby("week")["units"].mean().sort_index()
def index_to_pre(s):
    return s / s[s.index < CUT].mean() * 100
ti, ci = index_to_pre(t), index_to_pre(c)

# 5) Plot a readable window around the cut, smoothed 4 weeks for legibility.
lo, hi = CUT - pd.Timedelta(weeks=52), CUT + pd.Timedelta(weeks=40)
fig, ax = plt.subplots(figsize=(12, 4))
ti.rolling(4).mean().loc[lo:hi].plot(ax=ax, label=f"treated ({TREATED})", lw=2)
ci.rolling(4).mean().loc[lo:hi].plot(ax=ax, label=f"control avg (n={len(controls)})", lw=2)
ax.axvline(CUT, color="k", ls="--", lw=1, label="price cut")
ax.set_title("Weekly sales indexed to pre-cut mean (=100) — parallel LEFT of the line = DiD is credible")
ax.set_ylabel("index (pre-cut mean = 100)"); ax.legend(); plt.show()
# READ: do the lines track each other LEFT of the dashed line? If yes, parallel-trends holds and any
# post-cut GAP (treated rising above control) is the causal price effect. If they diverge pre-cut, this
# control is invalid -> pick different controls or a different treated item.

### 3d — A reusable screener with matched controls

The cell above rejected `FOODS_3_822`: (1) a 768-item control average is far smoother than one noisy item, so
"parallel" was unjudgeable, and (2) the item *crashed to zero at the cut* — a stockout, the opposite of a price
response. The screener below fixes the method so we can sweep candidates properly:

- **Matched controls** — instead of the whole department, keep the **top-K** same-store/same-dept items whose
  *pre-cut* weekly sales correlate most with the treated item. Comparable level and seasonality means both lines
  carry similar noise, so a divergence is real rather than an artefact of averaging.
- **A numeric parallel-trend score** — the mean pre-cut correlation of the matched set. High (say ≳0.5) = the
  controls genuinely co-move with the treated item before the cut; low = even the best controls don't track, so
  the DiD would be weak no matter how the plot looks.

It's a **function** — set `store/dept/treated/cut` and re-run to test any shortlist item. We start with
`CA_3 FOODS_2_227`; if its score is high, it doesn't crash at the cut, and a post-cut gap opens, that's our DiD.


## 6. Synthesis — the scope call (**SETTLED**)

**Forecasting scope:** FOODS_3 in **one store — CA_3**. Dense enough to model honestly (§2) and it houses the
chosen intervention. Realises the plan's "one store" lean, now evidenced.

**Causal intervention (DiD):** treated = **`FOODS_3_697` @ `CA_3`**, price cut ~**2011-08-06** ($3.58 → $2.98,
−17%). Density 42.2 units/wk, 0% zero-weeks. Controls = same-store/same-dept, matched on pre-cut weekly-sales
correlation (top-15, mean pre-cut corr **0.52**). Parallel pre-trend holds; clear sustained post-cut gap. Full
Feature-C plan (event-study, placebo, CI, five-store replication) lives in `SPEC.md` → Feature C.

**Key EDA facts driving these:**
- **Intermittency** heavy (overall median zero-share 0.73); FOODS densest, HOBBIES spikiest → *sparseness sets
  the model family* (LightGBM on dense FOODS) *and the metric* (RMSSE / WMAPE, never MAPE).
- **Trend** up; **weekly seasonality** strong (~+38% weekend) → seasonal-naive period-7 baseline, 7/28-day lags.
  Christmas closures = structural zeros → holiday flag (Feature A).
- **Prices:** 73% of store-items ever move; the penny-price (−99%) **artifact tail must be filtered**; clean promo
  band is 15–45%.
- **SNAP:** fixed ~10 days/month, **store-wide** → no within-store control → CausalImpact, not DiD.

**Caveats to carry forward:**
- The eyeballed price effect is **large** (elasticity ≈ −4/−5) → validate in Feature C, don't assert.
- **Structural (pre-launch) zeros inflate zero-share** → mask via price-row presence before modelling (Feature A).

**The rejection trail (kept on purpose):** `FOODS_3_822` (stockout coincided with the cut), `FOODS_2_227` (96%
zero-weeks — the 3b filter selected price-stability, not demand). Left in §3 as the honest iteration.

→ Feeds A0 and the subfeature runthrough.

### 3e — Fix the shortlist: add the demand bar

`FOODS_2_227` exposed the flaw: 3b filtered on *price-step* quality but never on whether the item **sells**. A
dead item's price is stable precisely because nobody buys it, so the shortlist filled with 90%+-zero series, and
indexing to their ~0 baseline produced the nonsense 80× spike. This is the §2 sparseness lesson again.

The fix intersects the clean-step list with a **demand bar** — ≤35% zero-weeks and ≥1 unit/day, computed straight
from the wide matrix — and requires the cut to sit ≥26 weeks inside the sales window (real history each side). It
ranks the survivors densest-first (which pulls us back toward FOODS_3, where the sellers are) and auto-screens the
top one. If that item crashes at the cut like FOODS_3_822 did, just call `screen_candidate()` on the next row.


In [ ]:
# --- Density-aware candidate ranking: intersect clean price-steps with items that ACTUALLY SELL. ---
# The 3b filter selected on price-step quality only, so it surfaced near-dead items (FOODS_2_227: 96%
# zero-weeks). Here we add the demand bar the §2 lesson demands, computed cheaply from the WIDE matrix.
dens = sales[["store_id", "item_id", "dept_id"]].copy()
dens["mean_daily"] = sales[d_cols].mean(axis=1)          # avg units/day over the whole history
dens["zero_share"] = (sales[d_cols] == 0).mean(axis=1)   # already seen in §2

SALES_START, SALES_END = cal_days.loc[d_cols[0]], cal_days.loc[d_cols[-1]]  # 2011-01-29 .. 2016-05-22
buffer = pd.Timedelta(weeks=26)  # need >=26w of real sales on each side of the cut

clean = runs[(runs["cat"] == "FOODS") &
             (runs["drop_pct"].between(-0.45, -0.15)) &
             (runs["prev_len"] >= 4) & (runs["run_len"] >= 4) &
             (runs["price"] >= 1.0)].merge(dens, on=["store_id", "item_id"], how="left")

dense = clean[(clean["zero_share"] <= 0.35) & (clean["mean_daily"] >= 1.0) &      # actually sells
              (clean["start"] > SALES_START + buffer) &
              (clean["start"] < SALES_END - buffer)].copy()                        # room both sides
dense["from_to"] = dense["prev_price"].round(2).astype(str) + " -> " + dense["price"].round(2).astype(str)
dense = dense.sort_values("zero_share")   # densest sellers first
print(f"{len(dense)} dense, clean, well-timed FOODS candidates (<=35% zero-weeks, >=1 unit/day)")
display(dense[["store_id", "item_id", "dept_id", "start", "from_to", "drop_pct",
               "mean_daily", "zero_share"]].head(15))

# Auto-screen the densest one; if it crashes at the cut, call screen_candidate() on the next row.
top = dense.iloc[0]
_ = screen_candidate(top["store_id"], top["dept_id"], top["item_id"], top["start"], k=15)

## 4. SNAP — feasibility for CausalImpact
*Days/month on, fixed per state? Visible aggregate lift on SNAP days? (Confirms SNAP -> CausalImpact, not DiD.)*

In [ ]:
snap_cols = ["snap_CA", "snap_TX", "snap_WI"]
print("share of days SNAP is ON:")
print(calendar[snap_cols].mean().round(3))
print("\nSNAP on-days per month (CA):")
print(calendar.groupby("month")["snap_CA"].sum().div(calendar['year'].nunique()).round(1))
# TODO: confirm the fixed ~10-days-per-month pattern, applied store-wide (=> no within-store control).

In [ ]:
# Aggregate lift on SNAP days, one state. FOODS is the SNAP-eligible category — filter to it for signal.
ca_stores = sales.loc[sales["state_id"] == "CA", :]
ca_foods = ca_stores.loc[ca_stores["cat_id"] == "FOODS", d_cols].sum(axis=0)
ca_foods.index = ca_foods.index.map(cal_days)
snap_flag = calendar.set_index("date")["snap_CA"].reindex(ca_foods.index)
print(ca_foods.groupby(snap_flag).mean().rename({0: "SNAP off", 1: "SNAP on"}).round(0))
# TODO: is the on/off gap real, or swamped by weekday effects (SNAP days cluster early-month)?

## 5. Events
*Which events, how frequent; any obvious spikes worth modelling as features vs ignoring?*

In [ ]:
print("days with a primary event:", calendar["event_name_1"].notna().sum(), "/", len(calendar))
print("days with a secondary event:", calendar["event_name_2"].notna().sum())
print("\nevent types:")
print(calendar["event_type_1"].value_counts(dropna=False))
# TODO: overlay a few big events (SuperBowl, Christmas, Thanksgiving) on daily_total to see spike magnitude.

## 6. Synthesis — the scope call

Record the decision here once the sections above are answered. This feeds the subfeature runthrough (A0).

- **One store vs one category?** (plan leans: one store, for clean same-store price-cut controls) — 
  _answer:_ ...
- **Which specific slice** — enough non-intermittent series *and* one clean datable price-cut with a usable control? — 
  _answer:_ ...
- **Anything that changes the plan** (e.g. intermittency worse than expected, price moves rarer than hoped)? — 
  _answer:_ ...